<a href="https://colab.research.google.com/github/blhuillier/2026B_GR/blob/main/ch04_index_gymnastics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 4 — Index gymnastics in Minkowski spacetime

This exercise is adapted from S. M. Carroll, *Spacetime and Geometry: An Introduction to General Relativity* (Addison-Wesley, 2004; Cambridge University Press, 2019), Exercise 1.7. We work with $\eta_{\mu\nu}=\mathrm{diag}(-1,1,1,1)$, as in the lecture notes.

Run the cell of helper functions below first. They lower and raise a chosen index, and take the symmetric and antisymmetric parts of a rank-two tensor.

For each part, **work out at least one component by hand first**, then use the code to get the rest and check yourself.

In [ ]:
try:
    import einsteinpy
except ImportError:
    !pip install einsteinpy
    import einsteinpy

import sympy as sp

# Initialize printing for nice LaTeX output
sp.init_printing(use_latex='mathjax')

In [ ]:
import sympy as sp
from sympy.tensor.array import tensorproduct, tensorcontraction, permutedims
from einsteinpy.symbolic import BaseRelativityTensor
from einsteinpy.symbolic.predefined import Minkowski
from IPython.display import display, Math


def lower_index(T, g, index_pos):
    """
    Lowers a specific index of a BaseRelativityTensor using a MetricTensor.
    index_pos is 0-indexed (e.g., 1 for the second index).
    """
    # 1. Multiply the tensor arrays
    prod_array = tensorproduct(T.tensor(), g.tensor())

    # 2. Contract the target index of T with the 1st index of g
    # In the product array, g's 1st index is located at len(T.config)
    g_first_idx = len(T.config)
    mixed_array = tensorcontraction(prod_array, (index_pos, g_first_idx))

    # The contraction leaves the new index LAST; move it back to index_pos.
    # Without this step, lowering any index but the last one transposes it.
    rank = len(T.config)
    order = list(range(rank - 1))
    order.insert(index_pos, rank - 1)
    mixed_array = permutedims(mixed_array, tuple(order))

    # 3. Update the configuration string (change 'u' to 'l' at index_pos)
    new_config = list(T.config)
    new_config[index_pos] = 'l'
    new_config = "".join(new_config)

    # 4. Wrap and return as a fully tracked relativity tensor
    return BaseRelativityTensor(mixed_array, syms=T.syms, config=new_config, parent_metric=g)


def raise_index(T, g, index_pos):
    """
    Raises a specific index of a BaseRelativityTensor using the inverse MetricTensor.
    index_pos is 0-indexed.
    """
    # 1. Compute the inverse metric array (g^{\mu\nu})
    # We use SymPy's exact matrix inversion for safety
    g_matrix = sp.Matrix(g.tensor())
    g_inv_array = sp.Array(g_matrix.inv())

    # 2. Multiply the tensor arrays
    prod_array = tensorproduct(T.tensor(), g_inv_array)

    # 3. Contract the target index of T with the 1st index of g_inv
    # In the product array, g_inv's 1st index is located at len(T.config)
    g_inv_first_idx = len(T.config)
    mixed_array = tensorcontraction(prod_array, (index_pos, g_inv_first_idx))

    # As in lower_index: move the new index from last place back to index_pos.
    rank = len(T.config)
    order = list(range(rank - 1))
    order.insert(index_pos, rank - 1)
    mixed_array = permutedims(mixed_array, tuple(order))

    # 4. Update the configuration string (change 'l' to 'u' at index_pos)
    new_config = list(T.config)
    new_config[index_pos] = 'u'
    new_config = "".join(new_config)

    # 5. Wrap and return as a fully tracked relativity tensor
    return BaseRelativityTensor(mixed_array, syms=T.syms, config=new_config, parent_metric=g)


def symmetrize(T):
    """
    Symmetrizes a rank-2 BaseRelativityTensor: 1/2 * (T + T^T).
    """
    # 1. Extract the underlying SymPy array
    T_array = T.tensor()

    # 2. Transpose the array by swapping the 0th and 1st indices
    T_array_T = permutedims(T_array, (1, 0))

    # 3. Perform the symbolic addition and multiply by exactly 1/2
    T_sym_array = sp.Rational(1, 2) * (T_array + T_array_T)

    # 4. Wrap and return as a fully tracked relativity tensor
    return BaseRelativityTensor(
        T_sym_array,
        syms=T.syms,
        config=T.config,
        parent_metric=T.parent_metric
    )


def asymmetrize(T):
    """
    Anti-symmetrizes a rank-2 BaseRelativityTensor: 1/2 * (T - T^T).
    """
    # 1. Extract the underlying SymPy array
    T_array = T.tensor()

    # 2. Transpose the array by swapping the 0th and 1st indices
    T_array_T = permutedims(T_array, (1, 0))

    # 3. Perform the symbolic subtraction and multiply by exactly 1/2
    T_asym_array = sp.Rational(1, 2) * (T_array - T_array_T)

    # 4. Wrap and return as a fully tracked relativity tensor
    return BaseRelativityTensor(
        T_asym_array,
        syms=T.syms,
        config=T.config,
        parent_metric=T.parent_metric
    )

def display_tensor(T, pre="", post=""):
    """
    Extracts the tensor array and formats it safely based on its rank.
    Rank 1 or 2 tensors are displayed as geometric matrices.
    Higher-rank tensors are displayed as native nested arrays.
    """
    arr = T.tensor()
    rank = len(arr.shape)

    if rank <= 2:
        # Safe to use sp.Matrix for 1D or 2D tensors
        latex_repr = sp.latex(sp.Matrix(arr))
    else:
        # For rank 3 or higher, rely on SymPy's native Array rendering
        latex_repr = sp.latex(arr)

    full_string = f"{pre}{latex_repr}{post}"
    display(Math(full_string))

Run this cell to set up the metric and the two objects the exercise is about.

In [ ]:
minkowski_metric = Minkowski(c=sp.Integer(1))

X_matrix = [[2, 0, 1, -1], [-1, 0, 3, 2], [-1, 1, 0, 0], [-2, 1, 1, -2]]
X_tensor_uu = BaseRelativityTensor(
    X_matrix,
    syms=minkowski_metric.syms,
    config='uu',
    parent_metric=minkowski_metric
)
display_tensor(X_tensor_uu, pre=r'X^{\mu\nu} = ')

V_u = BaseRelativityTensor(
    [-1, 2, 0, -2],
    syms=minkowski_metric.syms,
    config='u',
    parent_metric=minkowski_metric
)
display_tensor(V_u, pre=r'V^{\mu} = ')

**(a)** Compute ${X^{\mu}}_{\nu}$.

**(b)** Compute ${X_{\mu}}^{\nu}$. Compare it with your answer to (a): why are they different?

**(c)** Compute the symmetric part $X^{(\mu\nu)}$.

**(d)** Compute $X_{[\mu\nu]}$, the antisymmetric part with **both indices down**. Check the sign of a spatial component such as $X_{[12]}$ by hand.

**(e)** Compute the trace ${X^{\lambda}}_{\lambda}$.

**(f)** Compute $V_\mu$, and then the scalar $V^\mu V_\mu$. Is $V^\mu$ timelike, spacelike or null?

**(g)** Compute $V_\mu X^{\mu\nu}$.